In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import phoenix as px
from phoenix.evals import OpenAIModel
from phoenix.experiments import run_experiment, evaluate_experiment
from phoenix.experiments.types import Example
from phoenix.experiments.evaluators import create_evaluator
from phoenix.otel import register
from helper import get_phoenix_endpoint
import pandas as pd
from datetime import datetime
import os
import nest_asyncio
nest_asyncio.apply()

In [3]:
from init_phoenix import init_phoenix
PROJECT_NAME = "trajectory-eval"

client, tool_calling_client, tracer = init_phoenix(project_name=PROJECT_NAME)


OpenTelemetry Tracing Details
|  Phoenix Project: trajectory-eval
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
from tutorial1_code import Agent
agent = Agent(client, tool_calling_client, tracer)

In [5]:
px_client = px.Client()

In [6]:
convergence_questions = [
    "What was the average quantity sold per transaction?",
    "What is the mean number of items per sale?", 
    "Calculate the typical quantity per transaction",
    "What's the mean transaction size in terms of quantity?",
    "On average, how many items were purchased per transaction?",
    "What is the average basket size per sale?",
    "Calculate the mean number of products per purchase",
    "What's the typical number of units per order?",
    "What is the average number of products bought per purchase?",
    "Tell me the mean quantity of items in a typical transaction",
    "How many items does a customer buy on average per transaction?",
    "What's the usual number of units in each sale?",
    "What is the typical amount of products per transaction?",
    "Show the mean number of items customers purchase per visit",
    "What's the average quantity of units per shopping trip?",
    "How many products do customers typically buy in one transaction?",
    "What is the standard basket size in terms of quantity?"
]

convergence_df = pd.DataFrame({
    'question': convergence_questions
})

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
dataset = px_client.upload_dataset(dataframe=convergence_df, 
                                   dataset_name=f"convergence_questions-{now}",
                                   input_keys=["question"])

📤 Uploading dataset...
💾 Examples uploaded: http://localhost:6006/datasets/RGF0YXNldDo3/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246Nw==


In [7]:
print(get_phoenix_endpoint())

http://localhost:6006/


In [8]:
# helper method to format the output returned by the task
def format_message_steps(messages):
    """
    Convert a list of message objects into a readable format that shows the steps taken.

    Args:
        messages (list): A list of message objects containing role, content, tool calls, etc.

    Returns:
        str: A readable string showing the steps taken.
    """
    steps = []
    for message in messages:
        role = message.get("role")
        if role == "user":
            steps.append(f"User: {message.get('content')}")
        elif role == "system":
            steps.append("System: Provided context")
        elif role == "assistant":
            if message.get("tool_calls"):
                for tool_call in message["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    steps.append(f"Assistant: Called tool '{tool_name}'")
            else:
                steps.append(f"Assistant: {message.get('content')}")
        elif role == "tool":
            steps.append(f"Tool response: {message.get('content')}")
    
    return "\n".join(steps)

In [9]:
def run_agent_and_track_path(example: Example) -> str:
    messages = [{"role": "user", "content": example.input.get("question")}]
    ret, messages = agent.run_agent(messages)
    return {"path_length": len(messages), "messages": format_message_steps(messages)}

In [10]:
experiment = run_experiment(dataset,
                            run_agent_and_track_path,
                            experiment_name="Convergence Eval",
                            experiment_description="Evaluating the convergence of the agent")

🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDo3/experiments
🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDo3/compare?experimentId=RXhwZXJpbWVudDoxMA==


running tasks |          | 0/17 (0.0%) | ⏳ 00:00<? | ?it/s

Running agent with messages: [{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}]
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}, {'role': 'system', 'content': '\nYou are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.\n'}]
{'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_dxZn1sY8ISAuM8c1ugYvvHnD', 'function': {'arguments': '{"prompt":"What was the average quantity sold per transaction?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}
Received response with tool calls: True
Starting tool calls span
lookup_sales_data
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}, {'role': 'system', 'content': '\nYou are a helpful assista

running tasks |▌         | 1/17 (5.9%) | ⏳ 00:13<03:36 | 13.54s/it

Running agent with messages: [{'role': 'user', 'content': "What's the mean transaction size in terms of quantity?"}]
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': "What's the mean transaction size in terms of quantity?"}, {'role': 'system', 'content': '\nYou are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.\n'}]
{'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_ZpRJsoKmrwTSuoBhNrouBdLF', 'function': {'arguments': '{"prompt":"What is the mean transaction size in terms of quantity?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}
Received response with tool calls: True
Starting tool calls span
lookup_sales_data
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': "What's the mean transaction size in terms of quantity?"}, {'role': 'system', 'content': '\nYou are a he

...

Running agent with messages: [{'role': 'user', 'content': 'Show the mean number of items customers purchase per visit'}]
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'Show the mean number of items customers purchase per visit'}, {'role': 'system', 'content': '\nYou are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.\n'}]
{'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_zhPTzPSl83tvvrHplUwtLJDb', 'function': {'arguments': '{"prompt":"Show the mean number of items customers purchase per visit."}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}
Received response with tool calls: True
Starting tool calls span
lookup_sales_data
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'Show the mean number of items customers purchase per visit'}, {'role': 'system', 'content':

running tasks |███████▋  | 13/17 (76.5%) | ⏳ 01:02<00:17 |  4.32s/it

Running agent with messages: [{'role': 'user', 'content': 'How many products do customers typically buy in one transaction?'}]
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'How many products do customers typically buy in one transaction?'}, {'role': 'system', 'content': '\nYou are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.\n'}]
{'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_bQNaQfOCsrWs7SQJgZA6LayB', 'function': {'arguments': '{"prompt":"How many products do customers typically buy in one transaction?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}
Received response with tool calls: True
Starting tool calls span
lookup_sales_data
Starting router call span
Making router call to OpenAI
[{'role': 'user', 'content': 'How many products do customers typically buy in one transaction?'}, {'role

running tasks |██████████| 17/17 (100.0%) | ⏳ 01:11<00:00 |  4.23s/it

✅ Task runs completed.

🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDo3/compare?experimentId=RXhwZXJpbWVudDoxMA==

Tasks Summary (07/08/25 12:35 PM +0200)
---------------------------------------
   n_examples  n_runs  n_errors
0          17      17         0


In [11]:
experiment.as_dataframe()

,output,input,example_id
run_id,,,
RXhwZXJpbWVudFJ1bjo0Mg==,"{'path_length': 6, 'messages': 'User: What was...",{'question': 'What was the average quantity so...,RGF0YXNldEV4YW1wbGU6MTAz
RXhwZXJpbWVudFJ1bjo0Mw==,"{'path_length': 6, 'messages': 'User: What is ...",{'question': 'What is the mean number of items...,RGF0YXNldEV4YW1wbGU6MTA0
RXhwZXJpbWVudFJ1bjo0NA==,"{'path_length': 6, 'messages': 'User: Calculat...",{'question': 'Calculate the typical quantity p...,RGF0YXNldEV4YW1wbGU6MTA1
RXhwZXJpbWVudFJ1bjo0NQ==,"{'path_length': 6, 'messages': 'User: What's t...",{'question': 'What's the mean transaction size...,RGF0YXNldEV4YW1wbGU6MTA2
RXhwZXJpbWVudFJ1bjo0Ng==,"{'path_length': 6, 'messages': 'User: On avera...","{'question': 'On average, how many items were ...",RGF0YXNldEV4YW1wbGU6MTA3
RXhwZXJpbWVudFJ1bjo0Nw==,"{'path_length': 6, 'messages': 'User: What is ...",{'question': 'What is the average basket size ...,RGF0YXNldEV4YW1wbGU6MTA4
RXhwZXJpbWVudFJ1bjo0OA==,"{'path_length': 6, 'messages': 'User: Calculat...",{'question': 'Calculate the mean number of pro...,RGF0YXNldEV4YW1wbGU6MTA5
RXhwZXJpbWVudFJ1bjo0OQ==,"{'path_length': 6, 'messages': 'User: What's t...",{'question': 'What's the typical number of uni...,RGF0YXNldEV4YW1wbGU6MTEw
RXhwZXJpbWVudFJ1bjo1MA==,"{'path_length': 6, 'messages': 'User: What is ...",{'question': 'What is the average number of pr...,RGF0YXNldEV4YW1wbGU6MTEx


## Evaluating the Path

In [12]:
outputs = experiment.as_dataframe()["output"].to_dict().values()

# Will include the user and system messages
optimal_path_length = min(output.get('path_length') for output in outputs if output and output.get('path_length') is not None)
print(f"The optimal path length is {optimal_path_length}")

The optimal path length is 6


In [13]:
@create_evaluator(name="Convergence Eval", kind="CODE")
def evaluate_path_length(output: str) -> float:
    if output and output.get("path_length"):
        return optimal_path_length/float(output.get("path_length"))
    else:
        return 0

In [14]:
experiment = evaluate_experiment(experiment,
                            evaluators=[evaluate_path_length])

🧠 Evaluation started.


running experiment evaluations |██████████| 17/17 (100.0%) | ⏳ 00:02<00:00 |  6.73it/s


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDo3/compare?experimentId=RXhwZXJpbWVudDoxMA==

Experiment Summary (07/08/25 12:37 PM +0200)
--------------------------------------------
          evaluator   n  n_scores  avg_score
0  Convergence Eval  17        17        1.0

Tasks Summary (07/08/25 12:35 PM +0200)
---------------------------------------
   n_examples  n_runs  n_errors
0          17      17         0
